In [1]:
import pandas as pd
import sqlite3

In [2]:
connection = sqlite3.connect('../data/checking-logs.sqlite')

In [3]:
test_query = '''
WITH first_visit AS (
    SELECT 
        uid,
        MIN(datetime(first_view_ts)) AS first_view_time
    FROM test
    WHERE labname != 'project1'
    GROUP BY uid
),
delta AS (
    SELECT 
        t.uid,
        AVG((julianday(datetime(deadlines.deadlines, 'unixepoch')) - julianday(t.first_commit_ts)) * 24) AS avg_diff,
        CASE 
            WHEN t.first_commit_ts < fv.first_view_time THEN 'before'
            ELSE 'after'
        END AS time_period
    FROM test t
    JOIN deadlines ON t.labname = deadlines.labs
    JOIN first_visit fv ON t.uid = fv.uid
    WHERE t.labname != 'project1'
    GROUP BY t.uid, time_period
)
SELECT 
    time_period AS time,
    AVG(avg_diff) AS avg_diff
FROM delta
GROUP BY time_period;
'''
test_results = pd.read_sql(test_query, connection)
test_results

,time,avg_diff
0,after,103.981208
1,before,66.679398


In [4]:
control_query = '''
WITH average_first_visit AS (
    SELECT 
        AVG(datetime(first_view_ts)) AS avg_first_view_time
    FROM test
    WHERE labname != 'project1'
),
delta AS (
    SELECT 
        c.uid,
        AVG((julianday(datetime(deadlines.deadlines, 'unixepoch')) - julianday(c.first_commit_ts)) * 24) AS avg_diff,
        CASE 
            WHEN c.first_commit_ts < afv.avg_first_view_time THEN 'before'
            ELSE 'after'
        END AS time_period
    FROM control c
    JOIN deadlines ON c.labname = deadlines.labs
    CROSS JOIN average_first_visit afv  -- Используем CROSS JOIN для получения одного значения
    WHERE c.labname != 'project1'
    GROUP BY c.uid, time_period
)
SELECT 
    time_period AS time,
    AVG(avg_diff) AS avg_diff
FROM delta
GROUP BY time_period;

'''
control_results = pd.read_sql(control_query, connection)
control_results

,time,avg_diff
0,after,91.803412


In [5]:
connection.close()